# BT4012 Bitcoin Fraud — V24 Final Candidate

The latest verified public best is **V23: 0.94888**. V24 uses 60% V12 and 40% equally mixed categorical experts, followed by the same 2% predecessor correction. The V24 public score is unknown.

All required source is embedded below. Attach `train.csv`, `test.csv`, `sample_submission.csv`, and `txs_edgelist.csv`. The default path trains the baseline models from the original known labels; the archive also includes a cache for exact fast reproduction.

In [ ]:
import importlib.util, subprocess, sys
required={"numpy":"numpy","pandas":"pandas","scipy":"scipy","sklearn":"scikit-learn","lightgbm":"lightgbm","catboost":"catboost"}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,"-m","pip","install",*missing])


In [ ]:
from pathlib import Path
import sys
SOURCE_FILES = {'train_v12.py': '"""V12: rank blend of V8 and recent-regime LightGBM experts."""\nimport argparse,hashlib,json,zipfile\nfrom pathlib import Path\nimport numpy as np,pandas as pd,lightgbm as lgb\nfrom sklearn.metrics import roc_auc_score\nfrom scipy.stats import rankdata\n\nFEATURES=[f\'feat_{i}\' for i in range(1,166) if i not in [100,101,103,136,139]]\nFOLDS=[(24,35),(26,35),(28,35),(30,35)]\nMEMORY_HORIZONS=(7,14)\nEXPERT_WEIGHT=.6\nREGIME_START=24\n\ndef build_model():\n return lgb.LGBMClassifier(objective=\'binary\',metric=\'auc\',learning_rate=.035,n_estimators=352,num_leaves=15,min_child_samples=55,colsample_bytree=1.,subsample=1.,subsample_freq=0,reg_lambda=6.,max_bin=127,random_state=4012,n_jobs=4,verbosity=-1)\n\ndef write_csv(frame,path):\n path=Path(path);tmp=path.with_name(path.name+\'.tmp\');frame.to_csv(tmp,index=False)\n check=pd.read_csv(tmp);assert len(check)==len(frame) and list(check.columns)==list(frame.columns)\n tmp.replace(path)\n\ndef load_data(path):\n path=Path(path);names=[\'train.csv\',\'test.csv\',\'sample_submission.csv\'];frames=[]\n if path.is_dir():\n  for n in names:\n   f=list(path.rglob(n))\n   if len(f)!=1:raise ValueError(f\'Expected one {n}; found {len(f)}\')\n   frames.append(pd.read_csv(f[0]))\n else:\n  with zipfile.ZipFile(path) as z:\n   for n in names:\n    f=[x for x in z.namelist() if Path(x).name==n and not x.startswith(\'__MACOSX\')]\n    if len(f)!=1:raise ValueError(f\'Expected one {n}\')\n    with z.open(f[0]) as stream:frames.append(pd.read_csv(stream))\n return tuple(frames)\n\ndef predict(source,y,target,source_steps,model_dir=None):\n source=source[FEATURES].astype(\'float32\');target=target[FEATURES].astype(\'float32\');steps=np.asarray(source_steps);y=np.asarray(y,dtype=int)\n pred={};v4=None\n for name,params,sw in [(\'v4\',{},None),(\'class_weight_1p6\',{},np.where(y==1,1.6,1.)),(\'goss\',dict(data_sample_strategy=\'goss\',n_estimators=500),None)]:\n  m=build_model().set_params(**params).fit(source,y,sample_weight=sw);pred[name]=m.predict_proba(target)[:,1]\n  if name==\'v4\':v4=m\n  if model_dir is not None:m.booster_.save_model(str(model_dir/f\'{name}.txt\'))\n gain=v4.booster_.feature_importance(importance_type=\'gain\');order=np.argsort(-gain,kind=\'stable\');selected=[FEATURES[i] for i in order[:80]]\n top=build_model().fit(source[selected],y);pred[\'top80\']=top.predict_proba(target[selected])[:,1]\n if model_dir is not None:\n  top.booster_.save_model(str(model_dir/\'top80.txt\'));(model_dir/\'selected_features.json\').write_text(json.dumps(selected,indent=2))\n p5=.75*pred[\'v4\']+.25*pred[\'class_weight_1p6\'];p8=.75*p5+.25*(.5*pred[\'goss\']+.5*pred[\'top80\'])\n max_step=int(steps.max());active=max_step>=REGIME_START\n if active:\n  recent={}\n  for h in MEMORY_HORIZONS:\n   use=steps>max_step-h;x=source.loc[use];yr=y[use]\n   if int(use.sum())<1000:raise ValueError(f\'Insufficient known labels in recent {h}-step window\')\n   r0=build_model().fit(x,yr);r1=build_model().fit(x,yr,sample_weight=np.where(yr==1,1.6,1.))\n   recent[h]=.75*r0.predict_proba(target)[:,1]+.25*r1.predict_proba(target)[:,1]\n   if model_dir is not None:\n    r0.booster_.save_model(str(model_dir/f\'recent{h}.txt\'));r1.booster_.save_model(str(model_dir/f\'recent{h}_class_weight_1p6.txt\'))\n  rank=lambda p: rankdata(p,method=\'average\')/len(p)\n  p10=.4*rank(p8)+.3*rank(recent[7])+.3*rank(recent[14])\n else:p10=p8.copy()\n diag=dict(regime_experts_active=active,known_through_step=max_step,recent7_rows=int((steps>max_step-7).sum()),recent14_rows=int((steps>max_step-14).sum()),expert_weight=EXPERT_WEIGHT,memory_horizons=list(MEMORY_HORIZONS))\n return p8,p10,diag\n\ndef run(data_path,out_dir,validate=False):\n out=Path(out_dir);out.mkdir(parents=True,exist_ok=True);tr,te,sa=load_data(data_path);known=tr.label.notna().values;y=tr.label.values;s=tr.time_step.values\n assert tr.label.dropna().isin([0,1]).all() and s.max()<te.time_step.min()\n assert tr.txId.is_unique and te.txId.is_unique and not set(tr.txId)&set(te.txId)\n X=tr[FEATURES].astype(\'float32\');Xt=te[FEATURES].astype(\'float32\')\n assert np.isfinite(X.values).all() and np.isfinite(Xt.values).all()\n if validate:\n  rows=[]\n  for cut,end in FOLDS:\n   if cut<REGIME_START:continue\n   ti=np.flatnonzero(known&(s<=cut));vi=np.flatnonzero(known&(s>cut)&(s<=end))\n   p8,p10,diag=predict(X.iloc[ti],y[ti].astype(int),X.iloc[vi],s[ti]);a8=roc_auc_score(y[vi],p8);a10=roc_auc_score(y[vi],p10)\n   rows.append(dict(train_end=cut,valid_start=cut+1,valid_end=end,rows=len(vi),regime_experts_active=diag[\'regime_experts_active\'],v8_auc=a8,v12_auc=a10,delta=a10-a8))\n   write_csv(pd.DataFrame(dict(row=vi,step=s[vi],y=y[vi],v8=p8,v12=p10)),out/f\'validation_predictions_cut{cut}.csv\')\n   write_csv(pd.DataFrame(rows),out/\'validation_v12.csv\');print(\'VALIDATION\',rows[-1],flush=True)\n models=out/\'models\';models.mkdir(exist_ok=True);ti=np.flatnonzero(known);p8,p10,diag=predict(X.iloc[ti],y[ti].astype(int),Xt,s[ti],models)\n for name,p in [(\'submission_v12\',p10),(\'v8_reproduced_for_audit\',p8)]:\n  sub=sa[[\'index\']].merge(pd.DataFrame({\'index\':te[\'index\'],\'target\':p}),on=\'index\',how=\'left\',sort=False,validate=\'one_to_one\')\n  assert len(sub)==len(te) and sub.target.notna().all() and sub.target.between(0,1).all() and np.array_equal(sub[\'index\'],sa[\'index\'])\n  write_csv(sub,out/f\'{name}.csv\')\n meta=dict(recipe=\'when known training reaches step 24, rank-blend 40% V8 + 30% recent-7 model + 30% recent-14 model; otherwise V8\',known_labels=int(known.sum()),unknown_labels_excluded=int((~known).sum()),prediction_rows=len(te),features=FEATURES,regime_start=REGIME_START,final_training_diagnostics=diag,future_labels_used=False,unknown_labels_pseudo_labeled=False,leaderboard_feedback_informed_model_selection=True,public_score_used_for_parameter_fitting=False,validation_windows_reused_and_overlapping=True,lightgbm_version=lgb.__version__,numpy_version=np.__version__,pandas_version=pd.__version__,submission_sha256=hashlib.sha256((out/\'submission_v12.csv\').read_bytes()).hexdigest(),v8_sha256=hashlib.sha256((out/\'v8_reproduced_for_audit.csv\').read_bytes()).hexdigest())\n (out/\'run_metadata_v12.json\').write_text(json.dumps(meta,indent=2));print(\'FINAL\',meta,flush=True)\n return pd.read_csv(out/\'submission_v12.csv\')\n\ndef find_data():\n for root in [Path(\'/kaggle/input\'),Path.cwd()/\'data\',Path.cwd()]:\n  if root.exists():\n   for p in sorted(root.rglob(\'train.csv\')):\n    if (p.parent/\'test.csv\').exists() and (p.parent/\'sample_submission.csv\').exists():return p.parent\n raise FileNotFoundError(\'Attach the official competition dataset or pass --data\')\n\nif __name__==\'__main__\':\n ap=argparse.ArgumentParser();ap.add_argument(\'--data\');ap.add_argument(\'--out\',default=\'BT4012_V12_run\');ap.add_argument(\'--validate\',action=\'store_true\');a=ap.parse_args();run(a.data or find_data(),a.out,a.validate)\n', 'train_v19.py': '"""V19: 65/35 rank blend of V12 and a recent-14-step CatBoost model (1000 trees).\nRun: python train_v19.py --data /path/to/competition-data --out ./BT4012_V19_run\n"""\nimport argparse, hashlib, json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostClassifier\nfrom scipy.stats import rankdata\nimport train_v12\n\nFEATURES = train_v12.FEATURES\nCATEGORICAL = ["feat_3", "feat_112", "feat_113", "feat_148", "feat_149"]\nHORIZON = 14\nITERATIONS = 1000\n\ndef as_catboost_frame(frame):\n    x = frame[FEATURES].copy()\n    for col in CATEGORICAL:\n        x[col] = x[col].round(8).astype(str)\n    return x\n\ndef build_catboost():\n    return CatBoostClassifier(\n        iterations=ITERATIONS, depth=6, learning_rate=0.035,\n        loss_function="Logloss", eval_metric="AUC", l2_leaf_reg=6,\n        random_seed=4012, verbose=False, thread_count=4,\n        allow_writing_files=False,\n    )\n\ndef run(data_path, out_dir):\n    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)\n    tr, te, sample = train_v12.load_data(data_path)\n    known = tr.label.notna().to_numpy()\n    y = tr.loc[known, "label"].astype(int).to_numpy()\n    steps = tr.loc[known, "time_step"].to_numpy()\n    recent = steps > int(steps.max()) - HORIZON\n    model_dir = out / "models"; model_dir.mkdir(exist_ok=True)\n    v12_dir = model_dir / "v12"; v12_dir.mkdir(exist_ok=True)\n    _, p12, diag = train_v12.predict(\n        tr.loc[known, FEATURES], y, te[FEATURES], steps, v12_dir\n    )\n    if int(recent.sum()) < 1000:\n        raise ValueError("Too few known labels in the recent 14-step window")\n    model = build_catboost()\n    model.fit(as_catboost_frame(tr.loc[known, FEATURES].loc[recent]),\n              y[recent], cat_features=CATEGORICAL)\n    p_cat = model.predict_proba(as_catboost_frame(te[FEATURES]))[:, 1]\n    model.save_model(str(model_dir / "catboost_recent14_1000.cbm"))\n    rank = lambda p: rankdata(p, method="average") / len(p)\n    p19 = 0.65 * rank(p12) + 0.35 * rank(p_cat)\n    def submission(pred):\n        frame = sample[["index"]].merge(\n            pd.DataFrame({"index": te["index"], "target": pred}),\n            on="index", how="left", sort=False, validate="one_to_one")\n        assert len(frame) == len(te)\n        assert np.array_equal(frame["index"].to_numpy(), sample["index"].to_numpy())\n        assert frame.target.notna().all() and frame.target.between(0, 1).all()\n        return frame\n    outputs = {\n        "submission_v12.csv": submission(p12),\n        "submission_v19.csv": submission(p19),\n    }\n    for name, frame in outputs.items():\n        train_v12.write_csv(frame, out / name)\n    meta = {\n        "version": "V19",\n        "recipe": "65% rank(V12) + 35% rank(recent-14 categorical CatBoost, 1000 trees)",\n        "training_rows": int(known.sum()),\n        "unknown_labels_excluded": int((~known).sum()),\n        "recent14_labeled_rows": int(recent.sum()),\n        "prediction_rows": len(te),\n        "categorical_features": CATEGORICAL,\n        "catboost_iterations": ITERATIONS,\n        "catboost_depth": 6,\n        "catboost_l2_leaf_reg": 6,\n        "catboost_random_seed": 4012,\n        "public_scores": {"V16_fallback": 0.94832, "V18": 0.94801, "V19": 0.94845},\n        "validation_screen": "Across the four reused chronological cutoffs, 1000 trees exceeded 750 trees on each; mean blended AUC increased by about 0.000030. This is weak screening evidence, not a public-score guarantee.",\n        "future_labels_used": False,\n        "unknown_labels_pseudo_labeled": False,\n        "v12_diagnostics": diag,\n        "catboost_version": __import__("catboost").__version__,\n    }\n    for name in outputs:\n        meta[name + "_sha256"] = hashlib.sha256((out / name).read_bytes()).hexdigest()\n    (out / "run_metadata_v19.json").write_text(json.dumps(meta, indent=2))\n    print(json.dumps(meta, indent=2))\n    return outputs\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default=None)\n    ap.add_argument("--out", default="BT4012_V19_run")\n    args = ap.parse_args()\n    run(args.data or train_v12.find_data(), args.out)\n', 'graph_postprocess.py': '"""A small directional graph correction; no labels are read here."""\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import coo_matrix\nfrom scipy.stats import rankdata\n\n\ndef predecessor_correction(edges, txids, scores, alpha=0.02):\n    ids = pd.Index(np.asarray(txids))\n    scores = np.asarray(scores, dtype=float)\n    if not ids.is_unique or len(ids) != len(scores) or not np.isfinite(scores).all():\n        raise ValueError(\'IDs must be unique and aligned with finite scores\')\n    if not 0 <= alpha <= 1:\n        raise ValueError(\'alpha must be in [0, 1]\')\n    src = ids.get_indexer(edges.iloc[:, 0])\n    dst = ids.get_indexer(edges.iloc[:, 1])\n    keep = (src >= 0) & (dst >= 0)\n    # Row i contains predecessors j with an original edge j -> i.\n    incoming = coo_matrix((np.ones(int(keep.sum())), (dst[keep], src[keep])),\n                          shape=(len(ids), len(ids))).tocsr()\n    incoming.data[:] = 1  # Duplicate edges must not duplicate a neighbor\'s vote.\n    count = np.asarray(incoming.sum(axis=1)).ravel()\n    own_rank = rankdata(scores, method=\'average\') / len(scores)\n    neighbor_mean = np.divide(incoming @ own_rank, count,\n                              out=own_rank.copy(), where=count > 0)\n    corrected = (1-alpha)*own_rank + alpha*neighbor_mean\n    details = {\n        \'target_nodes\': len(ids),\n        \'target_to_target_edges\': int(incoming.nnz),\n        \'nodes_with_observed_predecessors\': int((count > 0).sum()),\n        \'predecessor_coverage\': float(np.mean(count > 0)),\n        \'alpha\': alpha,\n        \'direction\': \'predecessor\',\n        \'no_predecessor_rule\': \'retain own percentile rank\',\n    }\n    return corrected, details\n', 'train_v21.py': '"""V21: add an eight-category CatBoost expert to V20\'s original ensemble.\n\nFull reproduction: python train_v21.py --data /path/to/data --out ./V21_run\nReuse saved V19 predictions: add --baseline-dir baseline_predictions\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport zipfile\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostClassifier\nfrom scipy.stats import rankdata\nimport train_v12\nimport train_v19\nfrom graph_postprocess import predecessor_correction\n\nCATEGORIES=[\'feat_3\',\'feat_112\',\'feat_113\',\'feat_148\',\'feat_149\',\'feat_15\',\'feat_160\',\'feat_7\']\nITERATIONS=1500\nNEW_EXPERT_FRACTION=0.5\nGRAPH_ALPHA=0.02\nVERIFIED_V20_SHA256=\'7102c1c7f5f55294bcd390e75dc1666833cd5b86c90b73d5197894f8e42e65e7\'\n\n\ndef cat_frame(frame):\n    x=frame[train_v12.FEATURES].copy()\n    for col in CATEGORIES:x[col]=x[col].round(8).astype(str)\n    return x\n\n\ndef build_model():\n    return CatBoostClassifier(iterations=ITERATIONS,depth=6,learning_rate=.035,\n        loss_function=\'Logloss\',eval_metric=\'AUC\',l2_leaf_reg=6,\n        random_seed=4012,verbose=False,thread_count=4,allow_writing_files=False)\n\n\ndef load_edges(path):\n    path=Path(path)\n    if path.is_dir():\n        matches=list(path.rglob(\'txs_edgelist.csv\'))\n        if len(matches)!=1:raise ValueError(\'Expected exactly one official txs_edgelist.csv\')\n        return pd.read_csv(matches[0])\n    with zipfile.ZipFile(path) as archive:\n        matches=[n for n in archive.namelist() if Path(n).name==\'txs_edgelist.csv\' and not n.startswith(\'__MACOSX\')]\n        if len(matches)!=1:raise ValueError(\'Expected exactly one official txs_edgelist.csv\')\n        with archive.open(matches[0]) as stream:return pd.read_csv(stream)\n\n\ndef find_data():\n    return train_v12.find_data()\n\n\ndef run(data_path,out_dir,baseline_dir=None):\n    out=Path(out_dir);out.mkdir(parents=True,exist_ok=True)\n    tr,te,sample=train_v12.load_data(data_path)\n    if baseline_dir is None:\n        baseline_dir=out/\'baseline_v19\'\n        train_v19.run(data_path,baseline_dir)\n    baseline_dir=Path(baseline_dir)\n    pbase={}\n    cache=out/\'baseline_predictions\';cache.mkdir(exist_ok=True)\n    for key in [\'v12\',\'v19\']:\n        path=baseline_dir/f\'submission_{key}.csv\'\n        d=pd.read_csv(path)\n        if list(d.columns)!=[\'index\',\'target\'] or not d[\'index\'].is_unique:\n            raise ValueError(f\'Invalid baseline file: {path}\')\n        if len(d)!=len(te) or set(d[\'index\'])!=set(te[\'index\']):\n            raise ValueError(f\'Baseline indices do not match test rows: {path}\')\n        p=d.set_index(\'index\').loc[te[\'index\'],\'target\'].to_numpy()\n        if not np.isfinite(p).all() or not ((p>=0)&(p<=1)).all():\n            raise ValueError(\'Invalid baseline prediction values\')\n        pbase[key]=p\n        destination=cache/path.name\n        if path.resolve()!=destination.resolve():shutil.copyfile(path,destination)\n    known=tr.label.notna().to_numpy()\n    y=tr.loc[known,\'label\'].astype(int).to_numpy()\n    steps=tr.loc[known,\'time_step\'].to_numpy()\n    assert np.isin(y,[0,1]).all() and tr.time_step.max()<te.time_step.min()\n    recent=steps>int(steps.max())-14\n    if recent.sum()<1000:raise ValueError(\'Insufficient recent known labels\')\n    model=build_model()\n    model.fit(cat_frame(tr.loc[known].loc[recent]),y[recent],cat_features=CATEGORIES)\n    pc=model.predict_proba(cat_frame(te))[:,1]\n    model_dir=out/\'models\';model_dir.mkdir(exist_ok=True)\n    model.save_model(str(model_dir/\'catboost_categories8_recent14_1500.cbm\'))\n    # Exact selected recipe: half of the old categorical contribution is retained.\n    rank=lambda p:rankdata(p,method=\'average\')/len(p)\n    new_blend=.65*rank(pbase[\'v12\'])+.35*rank(pc)\n    pre_graph=(1-NEW_EXPERT_FRACTION)*pbase[\'v19\']+NEW_EXPERT_FRACTION*new_blend\n    edges=load_edges(data_path)\n    p21,graph=predecessor_correction(edges,te.txId.to_numpy(),pre_graph,GRAPH_ALPHA)\n    p20,_=predecessor_correction(edges,te.txId.to_numpy(),pbase[\'v19\'],GRAPH_ALPHA)\n    for name,p in [(\'submission_v21.csv\',p21),(\'submission_v20_fallback.csv\',p20)]:\n        sub=sample[[\'index\']].merge(pd.DataFrame({\'index\':te[\'index\'],\'target\':p}),\n            on=\'index\',how=\'left\',sort=False,validate=\'one_to_one\')\n        assert len(sub)==len(sample) and np.array_equal(sub[\'index\'],sample[\'index\'])\n        assert sub.target.notna().all() and sub.target.between(0,1).all()\n        train_v12.write_csv(sub,out/name)\n    # Save raw new-model predictions to support independent model-reload checks.\n    pd.DataFrame({\'index\':te[\'index\'],\'new_catboost_probability\':pc}).to_csv(out/\'new_expert_predictions.csv\',index=False)\n    sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()\n    fallback_hash=sha(out/\'submission_v20_fallback.csv\')\n    meta={\n        \'version\':\'V21\',\n        \'recipe\':\'half V19 + half (65% rank(V12) + 35% rank(eight-category 1500-tree CatBoost)), followed by the V20 2% predecessor correction\',\n        \'equivalent_pre_graph_weights\':{\'rank_V12\':.65,\'rank_original_CatBoost\':.175,\'rank_new_CatBoost\':.175},\n        \'categorical_features\':CATEGORIES,\'new_categorical_features\':[\'feat_15\',\'feat_160\',\'feat_7\'],\n        \'new_model_iterations\':ITERATIONS,\'new_model_depth\':6,\'new_model_learning_rate\':.035,\'new_model_l2_leaf_reg\':6,\'seed\':4012,\n        \'known_training_rows\':int(known.sum()),\'recent14_known_rows\':int(recent.sum()),\n        \'unknown_labels_excluded\':int((~known).sum()),\'prediction_rows\':len(te),\n        \'graph\':graph,\'candidate_public_auc\':None,\n        \'baseline_matches_verified_v20_bytes\':fallback_hash==VERIFIED_V20_SHA256,\n        \'baseline_public_auc\':.94859 if fallback_hash==VERIFIED_V20_SHA256 else None,\n        \'future_labels_used\':False,\'external_data_used\':False,\'pseudo_labels_used\':False,\n        \'candidate_sha256\':sha(out/\'submission_v21.csv\'),\'fallback_sha256\':fallback_hash,\n        \'validation_limitations\':\'Several overlapping chronological windows were reused during selection. Additional checks assess the fixed recipe, not an untouched final test. Public and private score gains remain unknown.\',\n    }\n    (out/\'run_metadata_v21.json\').write_text(json.dumps(meta,indent=2))\n    print(json.dumps(meta,indent=2),flush=True)\n    return pd.read_csv(out/\'submission_v21.csv\')\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--data\',default=None)\n    parser.add_argument(\'--out\',default=\'BT4012_V21_run\')\n    parser.add_argument(\'--baseline-dir\',default=None)\n    args=parser.parse_args()\n    run(args.data or find_data(),args.out,args.baseline_dir)\n', 'train_v22.py': '"""V22: combine V21 with a longer-trained recent categorical expert.\n\nFull reproduction: python train_v22.py --data /path/to/data --out ./V22_run\nReuse the included baseline cache: add --baseline-dir baseline_v21\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostClassifier\nfrom scipy.stats import rankdata\nimport train_v12\nimport train_v21\nfrom graph_postprocess import predecessor_correction\n\nITERATIONS=2000\nNEW_EXPERT_FRACTION=0.25\nGRAPH_ALPHA=0.02\nVERIFIED_V21_SHA256=\'df4b6c758af5b4c9d0181dd426380449965fca8e8fb85bcc08d7910c04302d8c\'\n\n\ndef make_model():\n    return CatBoostClassifier(iterations=ITERATIONS,depth=6,learning_rate=.035,\n        loss_function=\'Logloss\',eval_metric=\'AUC\',l2_leaf_reg=6,\n        random_seed=4012,verbose=False,thread_count=4,allow_writing_files=False)\n\n\ndef find_data():return train_v12.find_data()\n\n\ndef run(data_path,out_dir,baseline_dir=None):\n    out=Path(out_dir);out.mkdir(parents=True,exist_ok=True)\n    tr,te,sample=train_v12.load_data(data_path)\n    if baseline_dir is None:\n        baseline_dir=out/\'baseline_v21\'\n        train_v21.run(data_path,baseline_dir)\n    baseline_dir=Path(baseline_dir)\n    rank=lambda p:rankdata(p,method=\'average\')/len(p)\n    base={}\n    for name in [\'v12\',\'v19\']:\n        d=pd.read_csv(baseline_dir/f\'baseline_predictions/submission_{name}.csv\')\n        if not d[\'index\'].is_unique or len(d)!=len(te) or set(d[\'index\'])!=set(te[\'index\']):\n            raise ValueError(\'Invalid baseline prediction indices\')\n        base[name]=d.set_index(\'index\').loc[te[\'index\'],\'target\'].to_numpy()\n    model_path=baseline_dir/\'models/catboost_categories8_recent14_1500.cbm\'\n    old_model=CatBoostClassifier();old_model.load_model(str(model_path))\n    old_probability=old_model.predict_proba(train_v21.cat_frame(te),thread_count=4)[:,1]\n    pre21=.5*base[\'v19\']+.5*(.65*rank(base[\'v12\'])+.35*rank(old_probability))\n    edges=train_v21.load_edges(data_path)\n    baseline_prediction,_=predecessor_correction(edges,te.txId.to_numpy(),pre21,GRAPH_ALPHA)\n    baseline_csv=baseline_dir/\'submission_v21.csv\'\n    published=pd.read_csv(baseline_csv)\n    if not published[\'index\'].is_unique or len(published)!=len(te) or set(published[\'index\'])!=set(te[\'index\']):\n        raise ValueError(\'Invalid V21 baseline submission\')\n    baseline_error=float(np.max(np.abs(published.set_index(\'index\').loc[te[\'index\'],\'target\'].to_numpy()-baseline_prediction)))\n    if baseline_error>1e-12:raise ValueError(\'Baseline models and baseline CSV are inconsistent\')\n    known=tr.label.notna().to_numpy()\n    y=tr.loc[known,\'label\'].astype(int).to_numpy()\n    steps=tr.loc[known,\'time_step\'].to_numpy()\n    recent=steps>int(steps.max())-14\n    assert np.isin(y,[0,1]).all() and tr.time_step.max()<te.time_step.min()\n    if recent.sum()<1000:raise ValueError(\'Insufficient recent known labels\')\n    model=make_model()\n    model.fit(train_v21.cat_frame(tr.loc[known].loc[recent]),y[recent],cat_features=train_v21.CATEGORIES)\n    probability=model.predict_proba(train_v21.cat_frame(te))[:,1]\n    models=out/\'models\';models.mkdir(exist_ok=True)\n    model.save_model(str(models/\'catboost_categories8_recent14_2000.cbm\'))\n    expert_pre=.65*rank(base[\'v12\'])+.35*rank(probability)\n    pre22=(1-NEW_EXPERT_FRACTION)*pre21+NEW_EXPERT_FRACTION*expert_pre\n    p22,graph=predecessor_correction(edges,te.txId.to_numpy(),pre22,GRAPH_ALPHA)\n    sub=sample[[\'index\']].merge(pd.DataFrame({\'index\':te[\'index\'],\'target\':p22}),\n        on=\'index\',how=\'left\',sort=False,validate=\'one_to_one\')\n    assert len(sub)==len(sample) and np.array_equal(sub[\'index\'],sample[\'index\'])\n    assert sub.target.notna().all() and sub.target.between(0,1).all()\n    train_v12.write_csv(sub,out/\'submission_v22.csv\')\n    destination=out/\'submission_v21_fallback.csv\'\n    if baseline_csv.resolve()!=destination.resolve():shutil.copyfile(baseline_csv,destination)\n    pd.DataFrame({\'index\':te[\'index\'],\'new_expert_probability\':probability}).to_csv(out/\'new_expert_predictions.csv\',index=False)\n    # Exact float arrays support direct auditing without CSV round-trip rounding.\n    np.save(out/\'baseline_pre_graph.npy\',pre21)\n    np.save(out/\'new_pre_graph.npy\',pre22)\n    sha=lambda path:hashlib.sha256(path.read_bytes()).hexdigest()\n    fallback_hash=sha(destination)\n    meta={\n        \'version\':\'V22\',\'new_expert_iterations\':ITERATIONS,\n        \'new_expert_fraction\':NEW_EXPERT_FRACTION,\n        \'recipe\':\'blend V21 pre-graph predictions with 65% rank(V12) + 35% rank(new 2000-tree eight-category CatBoost), then apply the same 2% predecessor correction\',\n        \'categorical_features\':train_v21.CATEGORIES,\'depth\':6,\'learning_rate\':.035,\n        \'l2_leaf_reg\':6,\'seed\':4012,\'recent_horizon\':14,\n        \'known_training_rows\':int(known.sum()),\'recent14_known_rows\':int(recent.sum()),\n        \'unknown_labels_excluded\':int((~known).sum()),\'prediction_rows\':len(te),\n        \'baseline_reconstruction_max_error\':baseline_error,\'graph\':graph,\n        \'baseline_matches_verified_v21_bytes\':fallback_hash==VERIFIED_V21_SHA256,\n        \'baseline_public_auc\':.94880 if fallback_hash==VERIFIED_V21_SHA256 else None,\n        \'candidate_public_auc\':None,\'future_labels_used\':False,\n        \'external_data_used\':False,\'pseudo_labels_used\':False,\n        \'candidate_sha256\':sha(out/\'submission_v22.csv\'),\'fallback_sha256\':fallback_hash,\n        \'validation_limitations\':\'Chronological windows overlap and include reused data. Additional windows assess a fixed recipe, not an untouched final test. Public and private gains are unknown.\'\n    }\n    (out/\'run_metadata_v22.json\').write_text(json.dumps(meta,indent=2))\n    print(json.dumps(meta,indent=2),flush=True)\n    return sub\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--data\',default=None)\n    parser.add_argument(\'--out\',default=\'BT4012_V22_run\')\n    parser.add_argument(\'--baseline-dir\',default=None)\n    args=parser.parse_args()\n    run(args.data or find_data(),args.out,args.baseline_dir)\n', 'train_v23.py': '"""V23: combine the two proven eight-category experts with the V12 core.\n\nFull reproduction: python train_v23.py --data /path/to/data --out ./V23_run\nReuse the included cache: add --baseline-dir baseline_v22\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostClassifier\nfrom scipy.stats import rankdata\nimport train_v12\nimport train_v21\nimport train_v22\nfrom graph_postprocess import predecessor_correction\n\nGRAPH_ALPHA=0.02\nVERIFIED_V22_SHA256=\'02705e23b175c62a12eb1636d789682d9eb5b27836edfb109aba13bd752c815c\'\n\n\ndef find_data():return train_v12.find_data()\n\n\ndef run(data_path,out_dir,baseline_dir=None):\n    out=Path(out_dir);out.mkdir(parents=True,exist_ok=True)\n    if baseline_dir is None:\n        baseline_dir=out/\'baseline_v22\'\n        train_v22.run(data_path,baseline_dir)\n    baseline_dir=Path(baseline_dir)\n    _,te,sample=train_v12.load_data(data_path)\n    base_csv=baseline_dir/\'submission_v22.csv\'\n    baseline=pd.read_csv(base_csv)\n    if len(baseline)!=len(te) or not baseline[\'index\'].is_unique or set(baseline[\'index\'])!=set(te[\'index\']):\n        raise ValueError(\'Invalid baseline submission rows\')\n    pre22=np.load(baseline_dir/\'new_pre_graph.npy\')\n    if pre22.shape!=(len(te),):raise ValueError(\'Invalid baseline rank vector\')\n    edges=train_v21.load_edges(data_path)\n    reproduced,_=predecessor_correction(edges,te.txId.to_numpy(),pre22,GRAPH_ALPHA)\n    baseline_error=float(np.max(np.abs(reproduced-baseline.set_index(\'index\').loc[te[\'index\'],\'target\'].to_numpy())))\n    if baseline_error>1e-12:raise ValueError(\'Baseline cache does not match test row order\')\n    d=pd.read_csv(baseline_dir/\'baseline_v21/baseline_predictions/submission_v12.csv\')\n    if len(d)!=len(te) or not d[\'index\'].is_unique or set(d[\'index\'])!=set(te[\'index\']):\n        raise ValueError(\'Invalid V12 cached prediction rows\')\n    p12=d.set_index(\'index\').loc[te[\'index\'],\'target\'].to_numpy()\n    probability={}\n    paths={1500:baseline_dir/\'baseline_v21/models/catboost_categories8_recent14_1500.cbm\',\n           2000:baseline_dir/\'models/catboost_categories8_recent14_2000.cbm\'}\n    target=train_v21.cat_frame(te)\n    for count,path in paths.items():\n        model=CatBoostClassifier();model.load_model(str(path))\n        probability[count]=model.predict_proba(target,thread_count=4)[:,1]\n    rank=lambda p:rankdata(p,method=\'average\')/len(p)\n    pre23=.65*rank(p12)+.35*(.5*rank(probability[1500])+.5*rank(probability[2000]))\n    p23,graph=predecessor_correction(edges,te.txId.to_numpy(),pre23,GRAPH_ALPHA)\n    sub=sample[[\'index\']].merge(pd.DataFrame({\'index\':te[\'index\'],\'target\':p23}),\n        on=\'index\',how=\'left\',sort=False,validate=\'one_to_one\')\n    assert len(sub)==len(sample) and np.array_equal(sub[\'index\'],sample[\'index\'])\n    assert sub.target.notna().all() and sub.target.between(0,1).all()\n    train_v12.write_csv(sub,out/\'submission_v23.csv\')\n    fallback=out/\'submission_v22_fallback.csv\'\n    if base_csv.resolve()!=fallback.resolve():shutil.copyfile(base_csv,fallback)\n    np.save(out/\'new_pre_graph.npy\',pre23)\n    pd.DataFrame({\'index\':te[\'index\'],\'v12\':p12,\'cat8_1500\':probability[1500],\n                  \'cat8_2000\':probability[2000]}).to_csv(out/\'component_predictions.csv\',index=False)\n    sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()\n    fallback_hash=sha(fallback)\n    meta={\n        \'version\':\'V23\',\'weights_before_graph\':{\'rank_V12\':.65,\'rank_CatBoost8_1500\':.175,\'rank_CatBoost8_2000\':.175},\n        \'graph\':graph,\'baseline_reconstruction_max_error\':baseline_error,\n        \'baseline_matches_verified_v22_bytes\':fallback_hash==VERIFIED_V22_SHA256,\n        \'baseline_public_auc\':.94885 if fallback_hash==VERIFIED_V22_SHA256 else None,\n        \'candidate_public_auc\':None,\'new_pseudo_labels_used\':False,\'domain_weights_used\':False,\n        \'leaderboard_feedback_informed_selection\':True,\n        \'future_labels_used\':False,\'external_data_used\':False,\n        \'validation_summary\':\'Four of six reused historical windows improved. The two decreases were small; recent-period evidence is mixed. This candidate was chosen after rejecting several unreliable training experiments. Public and private gains are unknown.\',\n        \'candidate_sha256\':sha(out/\'submission_v23.csv\'),\'fallback_sha256\':fallback_hash,\n    }\n    (out/\'run_metadata_v23.json\').write_text(json.dumps(meta,indent=2))\n    print(json.dumps(meta,indent=2),flush=True)\n    return sub\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--data\',default=None)\n    parser.add_argument(\'--out\',default=\'BT4012_V23_run\')\n    parser.add_argument(\'--baseline-dir\',default=None)\n    args=parser.parse_args()\n    run(args.data or find_data(),args.out,args.baseline_dir)\n', 'train_v24.py': '"""Final V24: 60% V12 and 40% equally mixed categorical experts.\n\nFull reproduction: python train_v24.py --data /path/to/data --out ./V24_run\nFast reproduction: add --baseline-dir baseline_v23\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostClassifier\nfrom scipy.stats import rankdata\nimport train_v12\nimport train_v21\nimport train_v23\nfrom graph_postprocess import predecessor_correction\n\nCAT_WEIGHT=.40\nGRAPH_ALPHA=.02\nVERIFIED_V23_SHA256=\'e6850c2821d861ed293479abb960deba5ed3a4c220c38165a7338eb293eadf01\'\n\n\ndef find_data():return train_v12.find_data()\n\n\ndef run(data_path,out_dir,baseline_dir=None):\n    out=Path(out_dir);out.mkdir(parents=True,exist_ok=True)\n    if baseline_dir is None:\n        baseline_dir=out/\'baseline_v23\'\n        train_v23.run(data_path,baseline_dir)\n    baseline_dir=Path(baseline_dir)\n    _,te,sample=train_v12.load_data(data_path)\n    baseline_csv=baseline_dir/\'submission_v23.csv\'\n    baseline=pd.read_csv(baseline_csv)\n    if len(baseline)!=len(te) or not baseline[\'index\'].is_unique or set(baseline[\'index\'])!=set(te[\'index\']):\n        raise ValueError(\'Invalid baseline submission rows\')\n    baseline_pre=np.load(baseline_dir/\'new_pre_graph.npy\')\n    if baseline_pre.shape!=(len(te),):raise ValueError(\'Invalid baseline rank vector\')\n    edges=train_v21.load_edges(data_path)\n    baseline_reconstructed,_=predecessor_correction(edges,te.txId.to_numpy(),baseline_pre,GRAPH_ALPHA)\n    baseline_error=float(np.max(np.abs(baseline_reconstructed-baseline.set_index(\'index\').loc[te[\'index\'],\'target\'].to_numpy())))\n    if baseline_error>1e-12:raise ValueError(\'Baseline cache does not match test rows\')\n    cache=baseline_dir/\'baseline_v22\'\n    d=pd.read_csv(cache/\'baseline_v21/baseline_predictions/submission_v12.csv\')\n    if len(d)!=len(te) or not d[\'index\'].is_unique or set(d[\'index\'])!=set(te[\'index\']):\n        raise ValueError(\'Invalid cached V12 prediction rows\')\n    p12=d.set_index(\'index\').loc[te[\'index\'],\'target\'].to_numpy()\n    target=train_v21.cat_frame(te)\n    probs={}\n    paths={1500:cache/\'baseline_v21/models/catboost_categories8_recent14_1500.cbm\',\n           2000:cache/\'models/catboost_categories8_recent14_2000.cbm\'}\n    for count,path in paths.items():\n        model=CatBoostClassifier();model.load_model(str(path))\n        probs[count]=model.predict_proba(target,thread_count=4)[:,1]\n    rank=lambda p:rankdata(p,method=\'average\')/len(p)\n    categorical=.5*rank(probs[1500])+.5*rank(probs[2000])\n    pre=(1-CAT_WEIGHT)*rank(p12)+CAT_WEIGHT*categorical\n    pred,graph=predecessor_correction(edges,te.txId.to_numpy(),pre,GRAPH_ALPHA)\n    sub=sample[[\'index\']].merge(pd.DataFrame({\'index\':te[\'index\'],\'target\':pred}),\n        on=\'index\',how=\'left\',sort=False,validate=\'one_to_one\')\n    assert len(sub)==len(sample) and np.array_equal(sub[\'index\'],sample[\'index\'])\n    assert sub.target.notna().all() and sub.target.between(0,1).all()\n    train_v12.write_csv(sub,out/\'submission_v24.csv\')\n    fallback=out/\'submission_v23_fallback.csv\'\n    if baseline_csv.resolve()!=fallback.resolve():shutil.copyfile(baseline_csv,fallback)\n    np.save(out/\'new_pre_graph.npy\',pre)\n    pd.DataFrame({\'index\':te[\'index\'],\'v12\':p12,\'cat8_1500\':probs[1500],\n                  \'cat8_2000\':probs[2000]}).to_csv(out/\'component_predictions.csv\',index=False)\n    sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()\n    fallback_hash=sha(fallback)\n    meta={\n        \'version\':\'V24\',\'final_requested_submission\':True,\n        \'weights_before_graph\':{\'rank_V12\':.60,\'rank_CatBoost8_1500\':.20,\'rank_CatBoost8_2000\':.20},\n        \'graph\':graph,\'baseline_reconstruction_max_error\':baseline_error,\n        \'baseline_matches_verified_v23_bytes\':fallback_hash==VERIFIED_V23_SHA256,\n        \'baseline_public_auc\':.94888 if fallback_hash==VERIFIED_V23_SHA256 else None,\n        \'candidate_public_auc\':None,\'leaderboard_feedback_informed_selection\':True,\n        \'new_pseudo_labels_used\':False,\'domain_weights_used\':False,\n        \'future_labels_used\':False,\'external_data_used\':False,\n        \'selection_note\':\'A moderate increase to 40% categorical weight was fixed before an additional cutoff-26 check, which improved pooled and mean per-step AUC. Recent historical windows favored more categorical weight; an older stress window worsened. Results are mixed and reused, not an untouched final test.\',\n        \'candidate_sha256\':sha(out/\'submission_v24.csv\'),\'fallback_sha256\':fallback_hash,\n    }\n    (out/\'run_metadata_v24.json\').write_text(json.dumps(meta,indent=2))\n    print(json.dumps(meta,indent=2),flush=True)\n    return sub\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--data\',default=None)\n    parser.add_argument(\'--out\',default=\'BT4012_V24_run\')\n    parser.add_argument(\'--baseline-dir\',default=None)\n    args=parser.parse_args()\n    run(args.data or find_data(),args.out,args.baseline_dir)\n'}
CODE_DIR=Path.cwd()/"bt4012_v24_code"
CODE_DIR.mkdir(exist_ok=True)
for name,source in SOURCE_FILES.items():
    (CODE_DIR/name).write_text(source)
sys.path.insert(0,str(CODE_DIR.resolve()))


In [ ]:
from train_v24 import run, find_data
DATA_PATH = None  # optionally set the official data folder or ZIP path
BASELINE_DIR = None  # optionally point to the included baseline_v23 cache
OUTPUT_DIR=Path.cwd()/"BT4012_V24_output"
submission=run(DATA_PATH or find_data(),OUTPUT_DIR,BASELINE_DIR)
print("Submit:",OUTPUT_DIR/"submission_v24.csv")
submission.head()


## Evidence and fallback

The fixed 40% recipe improved an additional cutoff-26 check on pooled AUC and average AUC per step. Most recent historical windows favored it, but an older stress window worsened. The data and several windows were reused during selection; they are not independent final tests. Public or private leaderboard gains are unverified.

This is the final requested candidate. Keep `submission_v23_fallback.csv` (verified 0.94888) available if V24 does not improve the leaderboard score.